In [195]:
# SETUP LIBRARY DULU
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils import class_weight

In [87]:
# PERSIAPAN DATA DAN SCALING MODEL PERTAMA

data = pd.read_csv("../content/Final_data.csv")
data.head()

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,...,cal_from_macros,pct_carbs,protein_per_kg,pct_HRR,pct_maxHR,cal_balance,lean_mass_kg,expected_burn,Burns Calories (per 30 min)_bc,Burns_Calories_Bin
0,34.91,Male,65.27,1.62,188.58,157.65,69.05,1.00,1080.90,Strength,...,2139.59,0.500432,1.624789,0.741237,0.835985,725.10,47.777394,685.1600,7.260425e+19,Medium
1,23.37,Female,56.41,1.55,179.43,131.75,73.18,1.37,1809.91,HIIT,...,1711.65,0.500850,1.514093,0.551247,0.734270,-232.91,40.809803,978.6184,1.020506e+20,High
2,33.20,Female,58.98,1.67,175.04,123.95,54.96,0.91,802.26,Cardio,...,1965.92,0.500610,1.663445,0.574534,0.708124,805.74,44.635580,654.5266,1.079607e+20,High
3,38.69,Female,93.78,1.70,191.21,155.10,50.07,1.10,1450.79,HIIT,...,1627.28,0.499533,0.862017,0.744155,0.811150,1206.21,63.007432,773.6300,8.987921e+19,High
4,45.09,Male,52.42,1.88,193.58,152.88,70.84,1.08,1166.40,Strength,...,2659.23,0.500581,2.538153,0.668405,0.789751,303.60,43.347504,711.4176,5.264685e+19,Low


In [88]:
data.columns

Index(['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'Max_BPM', 'Avg_BPM',
       'Resting_BPM', 'Session_Duration (hours)', 'Calories_Burned',
       'Workout_Type', 'Fat_Percentage', 'Water_Intake (liters)',
       'Workout_Frequency (days/week)', 'Experience_Level', 'BMI',
       'Daily meals frequency', 'Physical exercise', 'Carbs', 'Proteins',
       'Fats', 'Calories', 'meal_name', 'meal_type', 'diet_type', 'sugar_g',
       'sodium_mg', 'cholesterol_mg', 'serving_size_g', 'cooking_method',
       'prep_time_min', 'cook_time_min', 'rating', 'Name of Exercise', 'Sets',
       'Reps', 'Benefit', 'Burns Calories (per 30 min)', 'Target Muscle Group',
       'Equipment Needed', 'Difficulty Level', 'Body Part', 'Type of Muscle',
       'Workout', 'BMI_calc', 'cal_from_macros', 'pct_carbs', 'protein_per_kg',
       'pct_HRR', 'pct_maxHR', 'cal_balance', 'lean_mass_kg', 'expected_burn',
       'Burns Calories (per 30 min)_bc', 'Burns_Calories_Bin'],
      dtype='object')

In [89]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 54 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Age                             20000 non-null  float64
 1   Gender                          20000 non-null  object 
 2   Weight (kg)                     20000 non-null  float64
 3   Height (m)                      20000 non-null  float64
 4   Max_BPM                         20000 non-null  float64
 5   Avg_BPM                         20000 non-null  float64
 6   Resting_BPM                     20000 non-null  float64
 7   Session_Duration (hours)        20000 non-null  float64
 8   Calories_Burned                 20000 non-null  float64
 9   Workout_Type                    20000 non-null  object 
 10  Fat_Percentage                  20000 non-null  float64
 11  Water_Intake (liters)           20000 non-null  float64
 12  Workout_Frequency (days/week)   

In [90]:
data.describe()

,Age,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Fat_Percentage,Water_Intake (liters),...,BMI_calc,cal_from_macros,pct_carbs,protein_per_kg,pct_HRR,pct_maxHR,cal_balance,lean_mass_kg,expected_burn,Burns Calories (per 30 min)_bc
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,...,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,2.000000e+04
mean,38.851453,73.898832,1.723093,179.889702,143.704306,62.195813,1.259446,1280.109600,26.101313,2.627485,...,24.921651,1998.297076,0.499983,1.460142,0.699005,0.802305,744.308699,53.786384,866.352318,8.631802e+19
std,12.114580,21.173010,0.127033,11.510805,14.267688,7.289351,0.341336,502.228982,4.996160,0.604724,...,6.701144,440.848408,0.001455,0.518946,0.144880,0.096613,720.946619,12.498740,250.317069,3.197579e+19
min,18.000000,39.180000,1.490000,159.310000,119.070000,49.490000,0.490000,323.110000,11.333134,1.460000,...,12.037907,1105.570000,0.492434,0.516706,0.371344,0.599789,-1266.220000,30.946261,219.852800,2.491905e+16
25%,28.170000,58.160000,1.620000,170.057500,131.220000,55.960000,1.050000,910.800000,22.387807,2.170000,...,20.094975,1661.022500,0.499054,1.076294,0.583656,0.727676,261.432500,44.587037,714.098250,6.441978e+19
50%,39.865000,70.000000,1.710000,180.140000,142.990000,62.200000,1.270000,1231.450000,25.822504,2.610000,...,24.119097,1943.130000,0.499981,1.382260,0.686284,0.794834,691.190000,51.204908,868.721400,8.371578e+19
75%,49.630000,86.100000,1.800000,189.425000,156.060000,68.090000,1.460000,1553.112500,29.676026,3.120000,...,28.562620,2271.950000,0.500910,1.750495,0.798196,0.869211,1176.290000,61.939016,1012.532700,1.100442e+20
max,59.670000,130.770000,2.010000,199.640000,169.840000,74.500000,2.020000,2890.820000,35.000000,3.730000,...,50.229544,3699.540000,0.507889,3.916881,1.073939,1.047032,3075.580000,90.117371,1477.108800,1.756614e+20


In [132]:
# CLEANING DATA
median_non_neg = data[data['Physical exercise'] >= 0]['Physical exercise'].median()
data['Physical exercise'] = data['Physical exercise'].apply(lambda x: median_non_neg if x < 0 else x)

data['Physical exercise'].describe()

,Physical exercise
count,20000.000000
mean,0.464266
std,0.981135
min,0.000000
25%,0.010000
50%,0.020000
75%,0.040000
max,4.050000


In [133]:
# ENCODING GENDER MENJADI BINER
data['Gender'] = data['Gender'].map({
    'Male' : 1,
    'Female' : 0
})

In [146]:
# MENDEFINISIKAN FITUR WAJIB (VARIABEL NUMERIK + VARIABEL KATEGORIKAL)
common_features = [
    'Age', 'Gender', 'Weight (kg)', 'Height (m)', 'BMI',
    'Workout_Frequency (days/week)', 'Session_Duration (hours)', 'Workout_Type',
    'Daily meals frequency', 'diet_type', 'Calories', 'Physical exercise'
]

available_features = [
    f for f in common_features if f in data.columns
]

X_common = data[availabe_features]
y_calories = data['Calories_Burned']

print("Fitur yang ada pada model pertama: \n",available_features)
print("\n✅ | Fitur utama yang digunakan totalnya ", len(available_features), available_features)

Fitur yang ada pada model pertama: 
 ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'BMI', 'Workout_Frequency (days/week)', 'Session_Duration (hours)', 'Workout_Type', 'Daily meals frequency', 'diet_type', 'Calories', 'Physical exercise']

✅ | Fitur utama yang digunakan totalnya  12 ['Age', 'Gender', 'Weight (kg)', 'Height (m)', 'BMI', 'Workout_Frequency (days/week)', 'Session_Duration (hours)', 'Workout_Type', 'Daily meals frequency', 'diet_type', 'Calories', 'Physical exercise']


In [142]:
# ONE-HOT ENCODING KATEGORI
onehot_cols_common = ['Workout_Type', 'diet_type']
df_common_encoded = X_common.copy()

if onehot_cols_common :
  ohe_common = OneHotEncoder(sparse_output=False, drop='first')
  ohe_result_common = ohe_common.fit_transform(df_common_encoded[onehot_cols_common])
  ohe_df_common = pd.DataFrame(ohe_result_common, columns=ohe_common.get_feature_names_out(onehot_cols_common))

  df_common_encoded = df_common_encoded.drop(columns = onehot_cols_common).reset_index(drop = True).join(ohe_df_common.reset_index(drop = True))

In [144]:
# STANDARD SCALLER
numeric_cols_common = df_common_encoded.select_dtypes(include=['int64','float64']).columns.tolist()
categorical_cols_common = df_common_encoded.columns.difference(numerix_cols_common)

scaler_common = StandardScaler()
scaled_numeric_common = scaler_common.fit_transform(df_common_encoded[numeric_cols_common])
scaled_df_common = pd.DataFrame(scaled_numeric_common, columns=numeric_cols_common)

In [145]:
# FINAL FEATURES
X_common_processed = df_common_encoded[categorical_cols_common].reset_index(drop = True).join(scaled_df_common.reset_index(drop = True))
print(f"✅ | Data diproses! Total fitur akhir {X_common_processed.shape[1]}")

✅ | Data diproses! Total fitur akhir 17


In [149]:
# MEMBAGI DATA TRAIN DENGAN DATA VALIDASI DAN DATA TEST
X_cal_train, X_cal_temp, y_cal_train, y_cal_temp = train_test_split(
    X_common_processed, y_calories,
    test_size=0.4,
    random_state=42
)

X_cal_val, X_cal_test, y_cal_val, y_cal_test = train_test_split(
    X_cal_temp, y_cal_temp,
    test_size=0.5,
    random_state=42
)

print("🙏 | Split data kalori (train/ validation/ test) selesai")

🙏 | Split data kalori (train/ validation/ test) selesai


In [150]:
# MENDEFINISIKAN EARLY STOPPING
early_stopping = EarlyStopping(
    monitor = 'val_mae',
    patience = 15,
    restore_best_weights = True
)

In [151]:
# MEMBANGUNG ARSITEKTUR MODEL PERTAMA
model_pertama = Sequential([
    Dense(64, activation = 'relu', input_shape = (X_cal_train.shape[1],)),
    Dropout(0.3),

    Dense(32, activation = 'relu'),
    Dropout(0.2),

    Dense(1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [159]:
# COMPILE MODEL DENGAN LEARNING RATE = 0.0001
model_pertama.compile(
    optimizer = Adam(learning_rate = 0.001),
    loss = 'mse',
    metrics = ['mae']
)

model_pertama.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 64)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,265 (12.75 KB)

 Trainable params: 3,265 (12.75 KB)

 Non-trainable params: 0 (0.00 B)

In [160]:
# TRAINING MODEL PERTAMA
print("\nTraining model pertama...")
history_cal = model_pertama.fit(
    X_cal_train, y_cal_train,
    validation_data = (X_cal_val, y_cal_val),
    epochs = 100,
    batch_size = 32,
    callbacks = [early_stopping],
    verbose = 1
)


Training model pertama...
Epoch 1/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 46353.1289 - mae: 164.8929 - val_loss: 5050.6934 - val_mae: 56.0738
Epoch 2/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 46485.7383 - mae: 164.2096 - val_loss: 4189.0098 - val_mae: 51.3090
Epoch 3/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 45420.5586 - mae: 162.9578 - val_loss: 4704.9253 - val_mae: 53.9897
Epoch 4/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 45740.0156 - mae: 163.6266 - val_loss: 4039.2959 - val_mae: 50.3596
Epoch 5/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 45345.3867 - mae: 162.2968 - val_loss: 4925.8672 - val_mae: 56.0661
Epoch 6/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 41766.4922 - mae: 157.6839 - val_loss: 4355.6538 - val_mae: 51.9387
Epoch 7/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 46087.1172 - mae: 162.3785 - val_loss: 4112.3857 - val_mae: 49.9795
Epoch 8/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 42942.757

In [161]:
# EVALUASI MODEL DENGAN DATA TEST
y_cal_pred = model_pertama.predict(X_cal_test)
mae_cal = mean_absolute_error(y_cal_test, y_cal_pred)
r2_cal = r2_score(y_cal_test, y_cal_pred)

print("\n ==== HASIL AKHIR MAE & R2 MODEL PERTAMA ====")
print(f"Mean Absolute Error (MAE): {mae_cal:.2f}")
print(f"R2 Score: {r2_cal:.4f}")

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step

 ==== HASIL AKHIR MAE & R2 MODEL PERTAMA ====
Mean Absolute Error (MAE): 35.87
R2 Score: 0.9913


In [164]:
# TAMBAH LAYER
model_pertama_tuned = Sequential([
    Dense(128, activation = 'relu', input_shape = (X_cal_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation = 'relu'),
    Dropout(0.2),
    Dense(32, activation = 'relu'),
    Dropout(0.2),
    Dense(1)
])

model_pertama_tuned.compile(
    optimizer = Adam(learning_rate = 0.001),
    loss = 'mse',
    metrics = ['mae']
)

model_dnn_pertama_tuned.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_9 (Dense)                 │ (None, 64)             │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,109 (27.77 KB)

 Trainable params: 2,369 (9.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 4,740 (18.52 KB)

In [167]:
history_cal_tuned = model_pertama_tuned.fit(
    X_cal_train, y_cal_train,
    validation_data = (X_cal_val, y_cal_val),
    epochs = 100,
    batch_size = 32,
    verbose = 2
)

y_cal_pred_tuned = model_pertama_tuned.predict(X_cal_test)
mae_cal_tuned = mean_absolute_error(y_cal_test, y_cal_pred_tuned)
mse_cal_tuned = mean_squared_error(y_cal_test, y_cal_pred_tuned)
r2_cal_tuned = r2_score(y_cal_test, y_cal_pred_tuned)

print("\n==== EVALUASI HASIL MODEL PERTAMA SETELAH DITUNING ====")
print(f"Mean Absolute Error (MAE): {mae_cal_tuned:.2f}")
print(f"Mean Squared Error (MSE): {mse_cal_tuned:.2f}")
print(f"R2 Score: {r2_cal_tuned:.4f}")


Epoch 1/100
375/375 - 2s - 6ms/step - loss: 36021.8477 - mae: 141.1859 - val_loss: 4481.8125 - val_mae: 51.7767
Epoch 2/100
375/375 - 2s - 5ms/step - loss: 37467.5703 - mae: 143.3808 - val_loss: 7644.3726 - val_mae: 69.9229
Epoch 3/100
375/375 - 1s - 4ms/step - loss: 35940.5859 - mae: 140.9026 - val_loss: 2362.2385 - val_mae: 37.5128
Epoch 4/100
375/375 - 1s - 3ms/step - loss: 34816.4336 - mae: 139.0045 - val_loss: 1970.9432 - val_mae: 34.7100
Epoch 5/100
375/375 - 1s - 3ms/step - loss: 34833.2812 - mae: 138.7968 - val_loss: 4045.8142 - val_mae: 49.8349
Epoch 6/100
375/375 - 1s - 3ms/step - loss: 35890.3906 - mae: 140.9886 - val_loss: 5464.7017 - val_mae: 57.1578
Epoch 7/100
375/375 - 1s - 3ms/step - loss: 35493.2734 - mae: 140.7077 - val_loss: 2365.2788 - val_mae: 38.2489
Epoch 8/100
375/375 - 1s - 3ms/step - loss: 34846.3164 - mae: 138.8690 - val_loss: 2644.7737 - val_mae: 40.4321
Epoch 9/100
375/375 - 1s - 3ms/step - loss: 34942.0664 - mae: 138.9185 - val_loss: 1819.2081 - val_mae: 

In [169]:
# PENGAMBILAN DATA UNTUK MODEL KEDUA
le = LabelEncoder()
data['Workout_Type_Encoded'] = le.fit_transform(data['Workout_Type'])
numeric_class = len(le.classes_)
y_cls_oh = to_categorical(data['Workout_Type_Encoded'])

# MENDEFINISIKAN FEATURES X_cls
cols_to_drop = [col for col in X_common_processed.columns if 'Workout_Type_' in col]
X_cls = X_common_processed.drop(columns = cols_to_drop)

print(f"✅ | Features Klasifikasi yang disiapkan, totalnya : {X_cls.shape[1]}")

✅ | Features Klasifikasi yang disiapkan, totalnya : 14


In [170]:
# MEMBAGI DATA
X_cls_train, X_cls_test, y_cls_train, y_cls_test = train_test_split(
    X_cls, y_cls_oh,
    test_size = 0.2,
    random_state = 42,
    stratify = data['Workout_Type_Encoded']
)

In [202]:
# MEMBANGUN ARSITEKTUR MODEL KEDUA
early_stopping_cls = EarlyStopping(
    monitor = 'val_accuracy',
    patience = 15,
    restore_best_weights = True
)

input_dim_cls = X_cls_train.shape[1]

model_kedua = Sequential([
    Dense(128, activation = 'relu', input_shape = (input_dim_cls,)),
    Dropout(0.3),

    Dense(64, activation = 'relu'),
    Dropout(0.2),

    Dense(numeric_class, activation = 'softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [204]:
# COMPILE MODEL KEDUA
model_kedua.compile(
    optimizer = Adam(learning_rate = 0.001),
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

model_kedua.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_40 (Dense)                │ (None, 128)            │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_23 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,436 (40.77 KB)

 Trainable params: 10,436 (40.77 KB)

 Non-trainable params: 0 (0.00 B)

In [201]:
# TRAINING MODEL KEDUA
y_train_labels = np.argmax(y_cls_train, axis = 1)
class_weights = class_weight.compute_class_weight(
    class_weight = 'balanced',
    classes = np.unique(y_train_labels),
    y = y_train_labels
)

class_weight_dict = dict(enumerate(class_weights))
print("Bobot kelas untuk model klasifikasi: ", class_weight_dict)

print("\nTraining model kedua ...")
history_cls = model_kedua.fit(
    X_cls_train, y_cls_train,
    epochs = 200,
    batch_size = 32,
    callbacks = [early_stopping_cls],
    class_weight = class_weight_dict,
    validation_split = 0.2,
    verbose = 1
)

Bobot kelas untuk model klasifikasi:  {0: np.float64(1.015744032503809), 1: np.float64(1.0052777079668258), 2: np.float64(0.9859502095144195), 3: np.float64(0.9935419771485345)}

Training model kedua ...
Epoch 1/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.2472 - loss: 1.4374 - val_accuracy: 0.2497 - val_loss: 1.3903
Epoch 2/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2487 - loss: 1.3939 - val_accuracy: 0.2528 - val_loss: 1.3875
Epoch 3/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2591 - loss: 1.3885 - val_accuracy: 0.2506 - val_loss: 1.3871
Epoch 4/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2693 - loss: 1.3864 - val_accuracy: 0.2428 - val_loss: 1.3873
Epoch 5/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2699 - loss: 1.3853 - val_accuracy: 0.2491 - val_loss: 1.3874
Epoch 6/200
400/400 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.2762 - loss: 1.3840 - val_accuracy: 0.2431 - val_loss: 1.3875
Epoch 7/200
400/400 ━━

In [205]:
# EVALUASI MODEL KEDUA
loss_cls, accuracy_cls = model_kedua.evaluate(X_cls_test, y_cls_test, verbose = 0)
print(f"\nLoss model kedua: {loss_cls:.4f}")
print(f"💀 | Akurasi model kedua: {accuracy_cls:.4f}")


Loss model kedua: 1.4710
💀 | Akurasi model kedua: 0.2570


In [208]:
# LAPORAN KLASIFIKASI
y_cls_pred_probs = model_kedua.predict(X_cls_test, verbose = 0)
y_cls_pred_probs_encoded= np.argmax(y_cls_pred_probs, axis = 1)

print("\n==== LAPORAN KLASIFIKASI ====")
print(classification_report(np.argmax(y_cls_test, axis = 1), y_cls_pred_probs_encoded, target_names = le.classes_))


==== LAPORAN KLASIFIKASI ====
              precision    recall  f1-score   support

      Cardio       0.25      0.31      0.28       985
        HIIT       0.27      0.20      0.23       995
    Strength       0.00      0.00      0.00      1014
        Yoga       0.26      0.52      0.34      1006

    accuracy                           0.26      4000
   macro avg       0.19      0.26      0.21      4000
weighted avg       0.19      0.26      0.21      4000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
